# Messlabor: Wie schnell sortiert der Streamingdienst?

Der Streamingdienst sortiert jede Nacht die Songs nach Streams. In diesem Notebook misst du, wie lange verschiedene Sortierverfahren dafür brauchen **und wie viele Vergleiche** sie machen, und vergleichst sie im Diagramm.

**So arbeitest du:**
- Lade dieses Notebook **und** die Datei `songs.csv` (Link auf der Kursseite) in den Dateibereich von JupyterLite.
- Teil 1 und 2 sind fertig: Du sagst vorher, führst aus und beantwortest die Fragen.
- Ab Teil 3 kommen deine Sortierverfahren dazu – immer dann, wenn du sie auf einer Kursseite kennengelernt hast.
- Alle Verfahren vergleichen zwei Streams **nur** mit `kommt_vor` und tauschen **nur** mit `tausche`. So zählt das Notebook jeden Vergleich mit.
- Lade das Notebook am Ende jeder Stunde herunter und arbeite beim nächsten Mal darin weiter. Führe dann zuerst alle Zellen von oben aus (Menü *Run → Run All Cells*).

| Teil | Kursseite | Inhalt |
|---|---|---|
| 1, 2 | 2.2.3 | Streams laden, Vergleichszähler, Stoppuhr |
| 3 | 2.2.3 | Bubblesort vervollständigen, Verdopplungsexperiment, dein eigenes Verfahren |
| 4 | 2.2.5 | Insertionsort und Selectionsort |
| 5 | 2.2.9 | Mergesort |
| 6 | 2.2.9b | Quicksort |

**Wenn etwas nicht klappt**
- `FileNotFoundError`: `songs.csv` fehlt im Dateibereich – hochladen und die Zelle noch einmal ausführen.
- `NameError: name '…' is not defined`: Eine Zelle weiter oben wurde noch nicht ausgeführt. Führe alle Zellen von oben aus.
- `IndentationError`: Beim Einfügen ist die Einrückung verrutscht. Jede Zeile innerhalb einer Funktion beginnt mit 4 Leerzeichen mehr als `def`.
- Die Liste ist nach dem Sortieren nicht sortiert, oder es werden 0 Vergleiche gezählt: Prüfe, ob dein Verfahren wirklich `kommt_vor` benutzt.
- `ZeroDivisionError` in der Faktor-Zelle: Es wurden 0 Vergleiche gezählt, dein Bubblesort ist noch nicht fertig.
- Eine Zelle rechnet sehr lange (Stern `[*]` links neben der Zelle): oben auf ■ klicken, um sie abzubrechen.

In [ ]:
import time
import matplotlib.pyplot as plt

## Teil 1: Die Streams laden – lesen und verstehen

Jede Zeile von `songs.csv` ist ein Song, die Spalten sind durch Semikolons getrennt:

```
titel;interpret;genre;jahr;dauer_s;streams
Sternengarten;Die Funkwellen;Hip-Hop;2013;174;5552
```

Zum Sortieren brauchen wir nur die Streams, die letzte Spalte (`teile[5]`). Das Einlesen kennst du aus 2.1.4 (Klausurpunkte).

In [ ]:
def lade_streams(dateiname):
    # liest die Streams (letzte Spalte) aller gültigen Songs
    streams = []
    with open(dateiname, "r", encoding="utf-8") as datei:
        datei.readline()                    # Kopfzeile überspringen
        for zeile in datei:
            zeile = zeile.strip()
            teile = zeile.split(";")
            if len(teile) == 6 and teile[5].isdigit():
                streams.append(int(teile[5]))
    return streams

streams = lade_streams("songs.csv")
print(len(streams), "Songs geladen")
print(streams[0:10])

**Verstehen**
1. Sage voraus, bevor du die Zelle ausführst: Stehen in `streams` Texte oder Zahlen? Woran erkennst du es an der Ausgabe?
2. Ganz am Ende von `songs.csv` stehen ein paar kaputte Zeilen, z. B. `Regenpuls;Mila Sander;Pop;2024;201;k.A.`. Welche Zeile im Code verhindert, dass das Programm dort abbricht? Was würde ohne sie passieren?

## Teil 2: Vergleichszähler und Stoppuhr – lesen und verstehen

Sekunden hängen vom Rechner ab, die Zahl der Vergleiche nur vom Verfahren und von den Daten. Deshalb messen wir beides.

- `kommt_vor(a, b)` sagt, ob der Stream `a` in der sortierten Liste vor `b` steht (aufsteigend), und zählt dabei jeden Vergleich in der Variablen `vergleiche` mit.
- `tausche(liste, i, j)` vertauscht die Einträge auf Platz `i` und `j`.
- `stoppe` misst, wie lange ein Sortierverfahren für eine Liste braucht.
- `messreihe` misst das für mehrere Listengrößen n hintereinander, immer mit den ersten n Streams, und trägt Zeit und Vergleiche in zwei Listen ein.

**Neu:** Eine Funktion kann man einer anderen Funktion als Parameter übergeben. Ohne Klammern ist `bubblesort` die Funktion selbst, erst `sortierfunktion(kopie)` führt sie aus. So funktioniert `stoppe` für jedes Sortierverfahren.

In [ ]:
vergleiche = 0      # zählt alle Vergleiche


def kommt_vor(a, b):
    # True, wenn a in der sortierten Liste vor b stehen soll (aufsteigend)
    global vergleiche
    vergleiche = vergleiche + 1
    return a < b


def tausche(liste, i, j):
    temp = liste[i]
    liste[i] = liste[j]
    liste[j] = temp


def stoppe(sortierfunktion, liste):
    # sortiert eine Kopie der Liste und gibt die Zeit in Sekunden zurück
    kopie = liste.copy()
    start = time.perf_counter()
    sortierfunktion(kopie)
    ende = time.perf_counter()
    return ende - start


def messreihe(sortierfunktion, groessen, zeiten, anzahl_vergleiche):
    # misst die Sortierfunktion für die ersten n Streams, für jedes n aus groessen,
    # und hängt Zeit und Anzahl der Vergleiche an die beiden Listen an
    global vergleiche
    for n in groessen:
        vergleiche = 0                                   # Zähler zurücksetzen
        zeit = stoppe(sortierfunktion, streams[0:n])     # streams aus Teil 1, die ersten n
        zeiten.append(zeit)
        anzahl_vergleiche.append(vergleiche)
        print(n, "Songs:", round(zeit, 3), "s,", vergleiche, "Vergleiche")

Sage vorher, was die nächste Zelle ausgibt, und führe sie dann aus.

In [ ]:
vergleiche = 0
print(kommt_vor(3, 5))
print(kommt_vor(5, 3))
print(kommt_vor(4, 4))
print("Vergleiche:", vergleiche)

**Verstehen**
1. Warum sortiert `stoppe` eine Kopie und nicht die Liste selbst? Denke an Teil 4: Dort wird dieselbe Liste `heute` nacheinander mit mehreren Verfahren gemessen.
2. `messreihe` gibt nichts zurück. Trotzdem stehen danach Werte in den Listen `zeiten` und `anzahl_vergleiche`. Erkläre das (2.1.6).
3. Was würde gemessen, wenn `messreihe` den Zähler vor jeder Messung nicht auf 0 zurücksetzt?

## Teil 3: Bubblesort mit Vergleichszähler (Seite 2.2.3)

Bubblesort kennst du aus 2.2.2. Hier vergleicht er **nur** mit `kommt_vor` und tauscht **nur** mit `tausche`.

Vervollständige Bubblesort: Wenn `liste[j + 1]` vor `liste[j]` gehört, werden die beiden getauscht.

In [ ]:
def bubblesort(liste):
    n = len(liste)
    for durchlauf in range(n - 1):
        getauscht = False
        for j in range(n - 1 - durchlauf):
            pass  # TODO: mit kommt_vor vergleichen, ggf. tauschen und getauscht auf True setzen
        if not getauscht:
            break
    return liste

Teste deine Funktion. Stimmt die Liste, aber die Zahl der Vergleiche nicht, dann vergleicht dein Bubblesort an einer Stelle ohne `kommt_vor`.

In [ ]:
test = [5, 1, 4, 2, 3]
vergleiche = 0
bubblesort(test)
print(test)                          # erwartet: [1, 2, 3, 4, 5]
print("Vergleiche:", vergleiche)     # erwartet: 9

### Das Verdopplungsexperiment

Sage vorher: Wenn sich n verdoppelt – wie verändert sich die Zeit, wie die Zahl der Vergleiche?

In [ ]:
groessen = [500, 1000, 2000, 4000]
zeiten_bubble = []
vergleiche_bubble = []
messreihe(bubblesort, groessen, zeiten_bubble, vergleiche_bubble)

In [ ]:
plt.plot(groessen, zeiten_bubble, marker="o", label="Bubblesort")
plt.xlabel("Anzahl der Songs n")
plt.ylabel("Zeit in Sekunden")
plt.title("Sortieren nach Streams: Zeit")
plt.legend()
plt.grid()
plt.show()

Das zweite Diagramm zeigt die Zahl der Vergleiche. Ergänze die Achsenbeschriftungen.

In [ ]:
plt.plot(groessen, vergleiche_bubble, marker="o", label="Bubblesort", color="navy")
plt.xlabel("")   # Beschriftung x-Achse
plt.ylabel("")   # Beschriftung y-Achse
plt.title("Sortieren nach Streams: Vergleiche")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Faktor bei Verdopplung: Wert für n geteilt durch Wert für die halbe Größe
# k = 1, 2, 3 ist die Position in der Liste groessen, k - 1 die Messung davor
for k in range(1, len(groessen)):
    faktor_zeit = zeiten_bubble[k] / zeiten_bubble[k - 1]
    faktor_vergleiche = vergleiche_bubble[k] / vergleiche_bubble[k - 1]
    print(groessen[k], "Songs: Faktor Zeit", round(faktor_zeit, 1), "  Faktor Vergleiche", round(faktor_vergleiche, 1))

In [ ]:
# gemessene Vergleiche und die Formel n · (n − 1) / 2
for k in range(len(groessen)):
    n = groessen[k]
    print(n, "Songs:", vergleiche_bubble[k], "gemessen,", n * (n - 1) // 2, "nach der Formel")

**Auswerten**
1. **Verdopplungsexperiment:** Welche Faktoren ergeben sich für die Zeit und für die Vergleiche? Triff eine Aussage zum Anstieg der Laufzeit. Warum schwankt der Faktor der Zeit stärker?
2. **Schritte statt Sekunden:** Vergleiche die gemessene Anzahl der Vergleiche mit n · (n − 1) / 2. Begründe, warum die Messung etwas darunter liegt.
3. Zu welcher Komplexitätsklasse gehört Bubblesort? Begründe mit den Diagrammen und den Faktoren.
4. **Hochrechnen:** `songs.csv` hat 50 000 Songs. Schätze mit deinem größten Messwert ab, wie lange Bubblesort für die ganze Liste bräuchte (nicht ausprobieren!). Trage den Messwert dann in den Hochrechner auf Seite 2.2.3 ein: Wie lange bräuchten die Charts für 100 Millionen Songs?

### Der beste Fall

Die Charts von gestern sind schon sortiert. Sage vorher, wie viele Vergleiche Bubblesort für 4 000 bereits sortierte Streams braucht, und führe die Zelle dann aus.

In [ ]:
sortiert = sorted(streams[0:4000])     # sorted liefert eine sortierte Kopie
vergleiche = 0
zeit = stoppe(bubblesort, sortiert)
print("Bubblesort, schon sortiert:", round(zeit, 4), "s,", vergleiche, "Vergleiche")

**Auswerten**
5. Erkläre die Zahl der Vergleiche mit dem Code von Bubblesort. Welche Komplexitätsklasse ergibt sich im besten Fall?

### Dein eigenes Verfahren aus 2.2.1

Füge dein Verfahren aus dem Kartenspiel in die nächste Zelle ein. Ersetze dazu die ganze Funktion (von `def` bis `pass`). Der Name muss `mein_verfahren` bleiben, sonst findet die Messung sie nicht. Sortiere **aufsteigend**.

**Wichtig:** Jeder Vergleich zweier Streams läuft über `kommt_vor`, jeder Tausch am besten über `tausche`. So übersetzt du:

| vorher | mit Zähler |
|---|---|
| `liste[a] < liste[b]` | `kommt_vor(liste[a], liste[b])` |
| `liste[a] > liste[b]` | `kommt_vor(liste[b], liste[a])` |
| drei Zeilen mit `temp` | `tausche(liste, a, b)` |

In [ ]:
# dein eigenes Verfahren aus 2.2.1
def mein_verfahren(liste):
    pass

Teste zuerst mit einer kleinen Liste. Gibt dein Verfahren eine neue Liste zurück, statt die übergebene zu sortieren, schreibe `test = mein_verfahren(test)`.

In [ ]:
test = [5, 1, 4, 2, 3]
vergleiche = 0
mein_verfahren(test)
print(test)                          # erwartet: [1, 2, 3, 4, 5]
print("Vergleiche:", vergleiche)

In [ ]:
zeiten_mein = []
vergleiche_mein = []
messreihe(mein_verfahren, groessen, zeiten_mein, vergleiche_mein)

plt.plot(groessen, vergleiche_bubble, marker="o", label="Bubblesort")
plt.plot(groessen, vergleiche_mein, marker="o", label="mein Verfahren")
plt.xlabel("Anzahl der Songs n")
plt.ylabel("Anzahl der Vergleiche")
plt.title("Sortieren nach Streams: Vergleiche")
plt.legend()
plt.grid()
plt.show()

**Auswerten**
6. Ändere die Faktor-Zelle oben so, dass sie die Faktoren für dein Verfahren ausgibt. Zu welcher Komplexitätsklasse gehört dein Verfahren?
7. Braucht dein Verfahren mehr oder weniger Vergleiche als Bubblesort? Ändert das etwas an der Klasse?

## Teil 4: Insertionsort und Selectionsort (Seite 2.2.5)

Füge Insertionsort und Selectionsort ein. Übersetze die Vergleiche wie in Teil 3 mit `kommt_vor`, die Tausche mit `tausche`. Teste beide und miss sie mit denselben Größen wie in Teil 3.

In [ ]:
# Insertionsort aus 2.2.5
def insertion_sort(liste):
    pass


# Selectionsort aus 2.2.5
def selection_sort(liste):
    pass

In [ ]:
test = [5, 1, 4, 2, 3]
insertion_sort(test)
print(test)                 # erwartet: [1, 2, 3, 4, 5]

test = [5, 1, 4, 2, 3]
selection_sort(test)
print(test)                 # erwartet: [1, 2, 3, 4, 5]

In [ ]:
zeiten_insertion = []
vergleiche_insertion = []
messreihe(insertion_sort, groessen, zeiten_insertion, vergleiche_insertion)

zeiten_selection = []
vergleiche_selection = []
messreihe(selection_sort, groessen, zeiten_selection, vergleiche_selection)

plt.plot(groessen, zeiten_bubble, marker="o", label="Bubblesort")
plt.plot(groessen, zeiten_mein, marker="o", label="mein Verfahren")
plt.plot(groessen, zeiten_insertion, marker="o", label="Insertionsort")
plt.plot(groessen, zeiten_selection, marker="o", label="Selectionsort")
plt.xlabel("Anzahl der Songs n")
plt.ylabel("Zeit in Sekunden")
plt.legend()
plt.grid()
plt.show()

**Die Charts von gestern:** 4 000 Songs sind schon sortiert, heute kommen 20 neue Songs hinten dazu. Sage voraus, welches Verfahren damit am schnellsten fertig ist und welches die meisten Vergleiche braucht. Miss dann.

In [ ]:
gestern = sorted(streams[0:4000])           # sorted liefert eine sortierte Kopie
heute = gestern + streams[4000:4020]        # 20 neue Songs hinten dran

vergleiche = 0
zeit = stoppe(insertion_sort, heute)
print("Insertionsort:", round(zeit, 3), "s,", vergleiche, "Vergleiche")

vergleiche = 0
zeit = stoppe(selection_sort, heute)
print("Selectionsort:", round(zeit, 3), "s,", vergleiche, "Vergleiche")

## Teil 5: Mergesort (Seite 2.2.9)

Füge `merge` und `mergesort` ein. In `merge` wird aus `links[i] <= rechts[j]` die Bedingung `not kommt_vor(rechts[j], links[i])` – so bleibt Mergesort stabil. Mergesort gibt eine neue Liste zurück – für die Zeitmessung ist das egal.

In [ ]:
# merge und mergesort aus 2.2.9
def merge(links, rechts):
    pass


def mergesort(liste):
    pass

In [ ]:
test = mergesort([5, 1, 4, 2, 3])
print(test)                 # erwartet: [1, 2, 3, 4, 5]

In [ ]:
zeiten_merge = []
vergleiche_merge = []
messreihe(mergesort, groessen, zeiten_merge, vergleiche_merge)

plt.plot(groessen, vergleiche_bubble, marker="o", label="Bubblesort")
plt.plot(groessen, vergleiche_insertion, marker="o", label="Insertionsort")
plt.plot(groessen, vergleiche_merge, marker="o", label="Mergesort")
plt.xlabel("Anzahl der Songs n")
plt.ylabel("Anzahl der Vergleiche")
plt.legend()
plt.grid()
plt.show()

Mergesort ist so schnell, dass wir die ganze Songliste nehmen können. Sage vorher: Was passiert mit der Zeit und mit der Zahl der Vergleiche, wenn sich n verdoppelt?

In [ ]:
groessen_gross = [12500, 25000, 50000]
zeiten_merge_gross = []
vergleiche_merge_gross = []
messreihe(mergesort, groessen_gross, zeiten_merge_gross, vergleiche_merge_gross)

plt.plot(groessen_gross, zeiten_merge_gross, marker="o", label="Mergesort")
plt.xlabel("Anzahl der Songs n")
plt.ylabel("Zeit in Sekunden")
plt.legend()
plt.grid()
plt.show()

**Auswerten:** Vergleiche die Zahl der Vergleiche für 50 000 Songs mit deiner Hochrechnung für Bubblesort aus Teil 3.

## Teil 6: Quicksort (Seite 2.2.9b)

Füge `partitioniere` und `quicksort(liste, links, rechts)` aus Schritt 4 von Seite 2.2.9b ein. `tausche` gibt es schon aus Teil 2, das brauchst du nicht noch einmal. In `partitioniere` wird aus `liste[j] < pivot` die Bedingung `kommt_vor(liste[j], pivot)`.

`stoppe` übergibt nur die Liste, deshalb gibt es die kleine Hülle `quicksort_ganz`, die die Grenzen ergänzt.

In [ ]:
# partitioniere und quicksort aus 2.2.9b
def partitioniere(liste, links, rechts):
    pass


def quicksort(liste, links, rechts):
    pass


def quicksort_ganz(liste):
    quicksort(liste, 0, len(liste) - 1)

In [ ]:
test = [5, 1, 4, 2, 3]
quicksort_ganz(test)
print(test)                 # erwartet: [1, 2, 3, 4, 5]

In [ ]:
zeiten_quick = []
vergleiche_quick = []
messreihe(quicksort_ganz, groessen, zeiten_quick, vergleiche_quick)

zeiten_quick_gross = []
vergleiche_quick_gross = []
messreihe(quicksort_ganz, groessen_gross, zeiten_quick_gross, vergleiche_quick_gross)

plt.plot(groessen_gross, zeiten_merge_gross, marker="o", label="Mergesort")
plt.plot(groessen_gross, zeiten_quick_gross, marker="o", label="Quicksort")
plt.xlabel("Anzahl der Songs n")
plt.ylabel("Zeit in Sekunden")
plt.legend()
plt.grid()
plt.show()

**Der Praxistest:** Wieder die Charts von gestern mit 20 neuen Songs, diesmal mit 20 000 Songs von gestern. Sage voraus, was Quicksort damit macht – und probiere es aus. Erkläre das Ergebnis. (Je nach Umgebung bricht Python mit `RecursionError` ab oder Quicksort braucht auffällig lange – beides hat dieselbe Ursache.)

In [ ]:
gestern = sorted(streams[0:20000])
heute = gestern + streams[20000:20020]      # 20 neue Songs hinten dran

vergleiche = 0
print("Insertionsort:", round(stoppe(insertion_sort, heute), 3), "s,", vergleiche, "Vergleiche")
vergleiche = 0
print("Mergesort:", round(stoppe(mergesort, heute), 3), "s,", vergleiche, "Vergleiche")
vergleiche = 0
print("Quicksort:", round(stoppe(quicksort_ganz, heute), 3), "s,", vergleiche, "Vergleiche")

## Zum Schluss: alle Verfahren im Vergleich

In [ ]:
plt.plot(groessen, zeiten_bubble, marker="o", label="Bubblesort")
plt.plot(groessen, zeiten_mein, marker="o", label="mein Verfahren")
plt.plot(groessen, zeiten_insertion, marker="o", label="Insertionsort")
plt.plot(groessen, zeiten_selection, marker="o", label="Selectionsort")
plt.plot(groessen, zeiten_merge, marker="o", label="Mergesort")
plt.plot(groessen, zeiten_quick, marker="o", label="Quicksort")
plt.xlabel("Anzahl der Songs n")
plt.ylabel("Zeit in Sekunden")
plt.title("Welches Verfahren nimmt der Streamingdienst?")
plt.legend()
plt.grid()
plt.show()

**Einschätzen:** Welches Verfahren würdest du dem Streamingdienst für die nächtlichen Charts empfehlen, welches für eine Playlist mit 30 Songs und welches, wenn nur ein paar neue Songs dazukommen? Begründe mit deinen Diagrammen.

## Zum Schluss: Zeit und Speicher

Bisher hast du nur gefragt, wie **lange** ein Verfahren braucht. Ein Verfahren braucht aber auch **Speicher**. Die Liste selbst zählt dabei nicht, sie muss ohnehin gespeichert werden. Es geht um den **zusätzlichen** Speicher, den das Verfahren während des Sortierens belegt. Er wird genauso in Abhängigkeit von n angegeben wie die Laufzeit.

Zusätzlicher Speicher entsteht auf zwei Wegen:
- **Neue Listen:** Wird ein Teil der Liste kopiert (z. B. `liste[:mitte]`) oder eine neue Ergebnisliste aufgebaut, belegt das Platz für jedes kopierte Element.
- **Offene Aufrufe:** Bei Rekursion merkt sich Python jeden Aufruf, der noch nicht fertig ist, mit seinen Variablen. Je tiefer die Rekursion, desto mehr Speicher. Sind es zu viele, bricht Python mit `RecursionError` ab. In Thonny liegt die Grenze bei etwa 1 000 offenen Aufrufen, JupyterLite erlaubt mehr.

Ein Verfahren, das nur in der Liste selbst tauscht und ein paar Hilfsvariablen wie `i`, `j` oder `temp` braucht, sortiert **in place**. Sein zusätzlicher Speicher hängt nicht von n ab: O(1).

**Auswerten**
1. Ergänze die Tabelle (Doppelklick auf diese Zelle). Für die Zeit helfen dir deine Messungen aus Teil 3 bis 6, für den Speicher der Code deiner Verfahren.

| Verfahren | Zeit im besten Fall | Zeit im schlechtesten Fall | zusätzlicher Speicher | Begründung für den Speicher |
|---|---|---|---|---|
| Bubblesort | | | | |
| Insertionsort | | | | |
| Selectionsort | | | | |
| Mergesort | | | | |
| Quicksort | | | | |

2. Gibt es ein Verfahren, das in **beiden** Spalten – schlechteste Zeit und Speicher – am besten abschneidet? Was folgt daraus für die Wahl eines Sortierverfahrens?
3. Wie tief wird die Rekursion bei Quicksort für 20 000 zufällige Streams ungefähr, wie tief für die Charts von gestern? Erkläre damit, warum Quicksort in Teil 6 bei den Charts von gestern so viele Vergleiche braucht und warum dasselbe Programm in Thonny mit `RecursionError` abbricht.
4. Der Streamingdienst hat 100 Millionen Songs. Die Streams belegen etwa 800 MB im Arbeitsspeicher. Wie viel Speicher braucht Mergesort zusätzlich, wie viel Quicksort? Wann würdest du trotz des Speichers Mergesort nehmen?